In [1]:
#Forward_pass is when the result remembers the events that created it 
# a = 2.0
# b = 3.0
# c = a * b ; The computer calculates c = 6.0, and immediately forgets how it got to 6.0. It doesn't remember that a and b were involved
# but here we want c to trace its steps backward to see exactly how changing a or changing b would affect that final result

In [2]:
class Value:
    # We give _children and _op default values so you don't have to type them every time
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children) # conversion to a set makes it easier to track unique parents
        self._op = _op
        self._backward = lambda: None  # A default empty function that does nothing

    def __repr__(self):
        # A __repr__ function MUST return the string
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        
        result_data = self.data + other.data  
        # Create a new Value object, passing the data, the parents, and the operation
        out = Value(result_data, _children=(self, other), _op='+')

        def _backward():
            # The children (self and other) gain gradient from the output's gradient
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        # Store this function inside our 'out' node so we can call it later
        out._backward = _backward
        return out

    def __mul__(self, other):
        result_data = self.data * other.data  
        out = Value(result_data, _children=(self, other), _op='*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
            
        # Store this function inside our 'out' node so we can call it later
        out._backward = _backward
        return out
    
    def backward(self):
        """
        Automates the entire backpropagation process by sorting the graph 
        topologically and executing the stored backward recipes in the correct order.
        """
        
        # We need to flatten our tree-like graph into a linear list where any 
        # parent node appears BEFORE its child nodes when looking from left to right.

        topo = []          # This will hold our final ordered list of nodes
        visited = set()    # Keeps track of nodes we've already processed to avoid infinite loops
        
        def build_topo(v):
            """
            A recursive helper function (Depth-First Search) that explores 
            the graph all the way to the starting input variables.
            """
            if v not in visited:
                visited.add(v)  # Mark this node as visited so we don't look at it again
                
                # First, recursively visit all the parent nodes that created this node
                for child in v._prev:
                    build_topo(child)
                    
                # Crucial Step: Only append this node to the list AFTER all of its
                # ancestors/parents have been completely explored and added.
                topo.append(v)
                
        # Kick off the recursive sorting starting from 'self' (the final output node)
        build_topo(self)
        
        # The derivative of the final output with respect to itself is always 1.0.
        # (e.g., d(Loss)/d(Loss) = 1.0). This kickstarts the entire math chain.

        self.grad = 1.0
        
        #  THE BACKWARD LOOP
        # 'topo' was built from inputs-to-output. We need to go BACKWARD 
        # from output-to-inputs. 'reversed(topo)' flips the list perfectly
        for node in reversed(topo):
            # Call the custom mathematical lambda or function we stored during the forward pass.
            # This calculates the local derivative and updates the gradients of its parents.
            node._backward()

In [3]:
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)

# Forward pass
d = a * b + c  # Result is 10.0

# ONE call triggers everything!
d.backward()

# Look at the final results!
print(f"a.grad: {a.grad}")  # Expected: 3.0
print(f"b.grad: {b.grad}")  # Expected: 2.0
print(f"c.grad: {c.grad}")  # Expected: 1.0

a.grad: 3.0
b.grad: 2.0
c.grad: 1.0
